<a href="https://colab.research.google.com/github/PauloSBLima/gado/blob/main/gado_v3.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
#%pip install ultralytics

In [ ]:
#%pip install opencv-python
#%pip install questionary

In [ ]:
#pip freeze > requirements.txt

#%pip install pipreqs
#!pipreqs . --encoding utf8 --force

In [ ]:
import cv2
import json
import time
import urllib.request
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
from ultralytics import YOLO
import os
import subprocess

# --- VERIFICAÇÃO DE AMBIENTE (COLAB OU LOCAL) ---
try:
    import google.colab
    IN_COLAB = True
    from google.colab.patches import cv2_imshow
    import ipywidgets as widgets
    from IPython.display import display, clear_output
except ImportError:
    IN_COLAB = False

arq_imagem = "gado001.jpg"

class MotorAgrotechYOLO:
    def __init__(self, nome_modelo='yolov8m.pt', numero_opcao=None, limiar_confianca=0.35):
        print(f"[SYSTEM] Inicializando Motor de Visão Computacional AgroTech com: {nome_modelo}")
        self.nome_modelo = nome_modelo
        self.numero_opcao = numero_opcao
        self.modelo = YOLO(nome_modelo)
        self.limiar_confianca = limiar_confianca
        
        self.classes_alvo, self.usa_world = self._configurar_classes_modelo()

    def _configurar_classes_modelo(self):
        if 'world' in self.nome_modelo.lower():
            print("[SYSTEM] Detectado modelo YOLO-World (Vocabulário Aberto). Configurando classes por texto...")
            alvos_texto = ['cattle', 'cow', 'bull', 'livestock', 'ox', 'horse', 'sheep']
            self.modelo.set_classes(alvos_texto)
            print(f"[SYSTEM] Classes de texto configuradas: {alvos_texto}")
            return alvos_texto, True
        else:
            print("[SYSTEM] Detectado modelo padrão/fechado. Mapeando IDs de classes...")
            alvos = ['cattle', 'cow', 'bull', 'livestock', 'ox', 'horse', 'sheep']
            ids_alvo = []
            
            for id_classe, nome_classe in self.modelo.names.items():
                if nome_classe in alvos:
                    ids_alvo.append(id_classe)
            
            if not ids_alvo and hasattr(self.modelo, 'names'):
                print("[AVISO] Nenhuma classe exata do agronegócio encontrada. Buscando termos alternativos...")
                for id_classe, nome_classe in self.modelo.names.items():
                    if nome_classe.lower() in alvos:
                        ids_alvo.append(id_classe)

            print(f"[SYSTEM] Classes de interesse mapeadas (IDs): {ids_alvo}")
            return ids_alvo, False

    def _desenhar_hud_profissional(self, img, contagem, tempo_ms, conf_media):
        overlay = img.copy()
        altura_painel = 80
        cv2.rectangle(overlay, (0, 0), (img.shape[1], altura_painel), (15, 15, 15), -1)
        cv2.addWeighted(overlay, 0.7, img, 0.3, 0, img)

        fonte = cv2.FONT_HERSHEY_SIMPLEX
        cv2.putText(img, f"AGROTECH ANALYTICS - MODELO: {self.nome_modelo.upper()}", (20, 30), fonte, 0.6, (0, 255, 0), 2)
        cv2.putText(img, f"Cabeças Detectadas: {contagem}", (20, 65), fonte, 0.8, (255, 255, 255), 2)
        cv2.putText(img, f"Latência: {tempo_ms:.1f}ms", (400, 65), fonte, 0.6, (200, 200, 200), 1)
        cv2.putText(img, f"Confiança Média: {conf_media:.1f}%", (600, 65), fonte, 0.6, (200, 200, 200), 1)
        return img

    def _plotar_grafico_confianca(self, payload_dados):
        if not payload_dados:
            print("[INFO] Nenhum dado para plotar no gráfico.")
            return

        dados_ordenados = sorted(payload_dados, key=lambda x: x["id_deteccao"])
        ids = [f"ID {d['id_deteccao']}\n({d['classe']})" for d in dados_ordenados]
        confiancas = [d['confianca_pct'] for d in dados_ordenados]

        plt.figure(figsize=(12, 5))
        barras = plt.bar(ids, confiancas, color='#2ecc71', edgecolor='#27ae60', width=0.6)

        limiar_pct = self.limiar_confianca * 100
        plt.axhline(y=limiar_pct, color='red', linestyle='--', linewidth=1.5, label=f'Limiar Mínimo ({limiar_pct:.1f}%)')

        plt.xlabel("Animais Detectados", fontsize=11, fontweight='bold')
        plt.ylabel("Nível de Confiança (%)", fontsize=11, fontweight='bold')
        plt.title(f"Métricas de Confiança - Modelo: {self.nome_modelo}", fontsize=12, fontweight='bold')
        plt.ylim(0, 105)
        plt.grid(axis='y', linestyle=':', alpha=0.7)
        plt.legend(loc='lower right')

        for barra in barras:
            altura = barra.get_height()
            plt.text(barra.get_x() + barra.get_width()/2., altura + 1.5, f"{altura:.1f}%",
                     ha='center', va='bottom', fontsize=9, fontweight='bold')

        plt.tight_layout()
        
        diretorio_graficos = Path("graficos")
        diretorio_graficos.mkdir(parents=True, exist_ok=True)
        nome_base_modelo = self.nome_modelo.rsplit('.', 1)[0]
        prefixo_nome = f"opt{self.numero_opcao}_{nome_base_modelo}" if self.numero_opcao else nome_base_modelo
        nome_arquivo_grafico = diretorio_graficos / f"{arq_imagem.rsplit('.', 1)[0]}_{prefixo_nome}_grafico_confianca.jpg"

        plt.savefig(str(nome_arquivo_grafico), dpi=300)
        print(f"[SUCESSO] Gráfico salvo em: {nome_arquivo_grafico}")
        plt.show()

    def processar_imagem(self, caminho_img):
        print(f"\n[ENGINE] Processando imagem: {caminho_img}")
        img = cv2.imread(str(caminho_img))
        if img is None:
            print(f"[ERRO] Falha ao carregar a imagem em: {caminho_img}")
            return

        inicio_tempo = time.time()
        args_predicao = {"source": img, "conf": self.limiar_confianca, "verbose": False}

        if not self.usa_world and self.classes_alvo:
            args_predicao["classes"] = self.classes_alvo

        resultados = self.modelo.predict(**args_predicao)
        tempo_inferencia = (time.time() - inicio_tempo) * 1000
        resultado = resultados[0]

        contagem = len(resultado.boxes)
        lista_confiancas = []
        payload_dados = []

        for i, box in enumerate(resultado.boxes):
            x1, y1, x2, y2 = box.xyxy[0].cpu().numpy().astype(int)
            confianca = float(box.conf[0].cpu().numpy()) * 100
            lista_confiancas.append(confianca)
            nome_classe = self.modelo.names[int(box.cls[0].item())]

            payload_dados.append({
                "id_deteccao": i + 1,
                "classe": nome_classe,
                "confianca_pct": round(confianca, 2),
                "geometria_bbox": {"x_min": int(x1), "y_min": int(y1), "x_max": int(x2), "y_max": int(y2)}
            })

            cv2.rectangle(img, (x1, y1), (x2, y2), (0, 255, 150), 2)
            texto = f"ID:{i+1} {nome_classe} {confianca:.1f}%"
            (w_txt, h_txt), _ = cv2.getTextSize(texto, cv2.FONT_HERSHEY_SIMPLEX, 0.5, 1)
            cv2.rectangle(img, (x1, y1 - 20), (x1 + w_txt, y1), (0, 255, 150), -1)
            cv2.putText(img, texto, (x1, y1 - 5), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 0, 0), 1)

        conf_media = sum(lista_confiancas) / len(lista_confiancas) if contagem > 0 else 0
        img_final = self._desenhar_hud_profissional(img, contagem, tempo_inferencia, conf_media)

        nome_base_modelo = self.nome_modelo.rsplit('.', 1)[0]
        prefixo_nome = f"opt{self.numero_opcao}_{nome_base_modelo}" if self.numero_opcao else nome_base_modelo

        Path("laudos").mkdir(parents=True, exist_ok=True)
        nome_arquivo_imagem = Path("laudos") / f"{arq_imagem.rsplit('.', 1)[0]}_{prefixo_nome}_laudo_agrotech.jpg"
        cv2.imwrite(str(nome_arquivo_imagem), img_final)

        Path("contagem").mkdir(parents=True, exist_ok=True)
        nome_arquivo_json = Path("contagem") / f"{arq_imagem.rsplit('.', 1)[0]}_{prefixo_nome}_dados_contagem.json"

        json_saida = {
            "metadata_lote": {
                "timestamp_processamento": time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime()),
                "modelo_utilizado": self.nome_modelo,
                "opcao_escolhida": self.numero_opcao,
                "total_cabecas_detectadas": contagem,
                "confianca_media_lote": round(conf_media, 2),
                "tempo_inferencia_ms": round(tempo_inferencia, 2)
            },
            "detalhamento_animais": payload_dados
        }

        with open(nome_arquivo_json, "w") as f:
            json.dump(json_saida, f, indent=4)

        print(f"[SUCESSO] Detecção concluída: {contagem} animais encontrados.")
        print(f"[SUCESSO] Laudo visual salvo em: {nome_arquivo_imagem}")
        print(f"[SUCESSO] Dados estruturados salvos em: {nome_arquivo_json}")

        escala = 0.5 
        largura = int(img_final.shape[1] * escala)
        altura = int(img_final.shape[0] * escala)
        img_redimensionada = cv2.resize(img_final, (largura, altura), interpolation=cv2.INTER_AREA)

        # ADAPTAÇÃO DINÂMICA DE EXIBIÇÃO
        if IN_COLAB:
            print("\n[EXIBIÇÃO] Renderizando imagem no Google Colab...")
            cv2_imshow(img_redimensionada)
        else:
            print("\n[EXIBIÇÃO] Abrindo janela com o resultado visual...")
            nome_janela = f"AgroTech Analytics - {self.nome_modelo}"
            cv2.namedWindow(nome_janela, cv2.WINDOW_NORMAL)
            cv2.imshow(nome_janela, img_redimensionada)
            cv2.waitKey(0)  
            cv2.destroyAllWindows() 

        self._plotar_grafico_confianca(payload_dados)

def executar_logica_execucao(opcao):
    mapa_modelos = {
        '1': 'yolov5n.pt', '2': 'yolov5s.pt', '3': 'yolov5m.pt', '4': 'yolov5l.pt', '5': 'yolov5x.pt',
        '6': 'yolov8n.pt', '7': 'yolov8s.pt', '8': 'yolov8m.pt', '9': 'yolov8l.pt', '10': 'yolov8x.pt',
        '11': 'yolov8n-oiv7.pt', '12': 'yolov8s-oiv7.pt', '13': 'yolov8m-oiv7.pt', '14': 'yolov8l-oiv7.pt', '15': 'yolov8x-oiv7.pt',
        '16': 'yolov9c.pt', '17': 'yolov9e.pt',
        '18': 'yolov10n.pt', '19': 'yolov10s.pt', '20': 'yolov10m.pt', '21': 'yolov10l.pt', '22': 'yolov10x.pt',
        '23': 'yolo11n.pt', '24': 'yolo11s.pt', '25': 'yolo11m.pt', '26': 'yolo11l.pt', '27': 'yolo11x.pt',
        '28': 'yolo12n.pt', '29': 'yolo12s.pt', '30': 'yolo12m.pt', '31': 'yolo12l.pt', '32': 'yolo12x.pt',
        '33': 'yolo26n.pt', '34': 'yolo26s.pt', '35': 'yolo26m.pt', '36': 'yolo26l.pt', '37': 'yolo26x.pt',
        '38': 'yolov8s-world.pt', '39': 'yolov8m-world.pt', '40': 'yolov8l-world.pt', '41': 'yolov8x-world.pt'
    }

    modelo_escolhido = mapa_modelos.get(str(opcao))

    if not modelo_escolhido:
        print("\n[AVISO] Opção inválida! O sistema usará o padrão por segurança: yolov8m-oiv7.pt (Opção 13)")
        modelo_escolhido = 'yolov8m-oiv7.pt'
        opcao = '13'
    else:
        print(f"\n[OK] Modelo selecionado com sucesso: {modelo_escolhido} (Opção {opcao})")

    diretorio_imagens = Path("imagens")
    diretorio_imagens.mkdir(parents=True, exist_ok=True)

    img_path = diretorio_imagens / arq_imagem
    
    if not img_path.exists():
        print(f"-> Baixando imagem aérea de rebanho para '{diretorio_imagens}/'...")
        url_imagem = "https://go.agenciasebrae.com.br/wp-content/uploads/sites/9/2024/10/Sodino-Fazenda-Nova-Carla-Gomes-1-scaled.jpg"
        req = urllib.request.Request(url_imagem, headers={'User-Agent': 'Mozilla/5.0'})
        with urllib.request.urlopen(req) as resposta, open(img_path, 'wb') as arquivo_saida:
            arquivo_saida.write(resposta.read())

    motor = MotorAgrotechYOLO(nome_modelo=modelo_escolhido, numero_opcao=str(opcao), limiar_confianca=0.20)
    motor.processar_imagem(img_path)

def executar_simulacao():
    if not IN_COLAB:
        comando_limpeza = 'cls' if os.name == 'nt' else 'clear'
        subprocess.run(comando_limpeza, shell=True)

    print("=" * 75)
    print("      SISTEMA DE CONTAGEM DE GADO - SELEÇÃO COMPLETA DE MODELOS YOLO")
    print("=" * 75)
    print(" Selecione a família e variação do modelo desejado:")
    print("  --- YOLOv5 ---")
    print("  [1] yolov5n.pt   | [2] yolov5s.pt   | [3] yolov5m.pt   | [4] yolov5l.pt   | [5] yolov5x.pt")
    print("  --- YOLOv8 ---")
    print("  [6] yolov8n.pt   | [7] yolov8s.pt   | [8] yolov8m.pt   | [9] yolov8l.pt   | [10] yolov8x.pt")
    print("  --- YOLOv8 OIV7 (Específico p/ Agro/Objetos Complexos) ---")
    print("  [11] yolov8n-oiv7.pt | [12] yolov8s-oiv7.pt | [13] yolov8m-oiv7.pt | [14] yolov8l-oiv7.pt | [15] yolov8x-oiv7.pt")
    print("  --- YOLOv9 ---")
    print("  [16] yolov9c.pt  | [17] yolov9e.pt")
    print("  --- YOLOv10 ---")
    print("  [18] yolov10n.pt | [19] yolov10s.pt | [20] yolov10m.pt | [21] yolov10l.pt | [22] yolov10x.pt")
    print("  --- YOLO11 ---")
    print("  [23] yolo11n.pt  | [24] yolo11s.pt  | [25] yolo11m.pt  | [26] yolo11l.pt  | [27] yolo11x.pt")
    print("  --- YOLO12 ---")
    print("  [28] yolo12n.pt  | [29] yolo12s.pt  | [30] yolo12m.pt  | [31] yolo12l.pt  | [32] yolo12x.pt")
    print("  --- YOLO26 (Mais Atual / Estado da Arte) ---")
    print("  [33] yolo26n.pt  | [34] yolo26s.pt  | [35] yolo26m.pt  | [36] yolo26l.pt  | [37] yolo26x.pt")
    print("  --- YOLO-World (Vocabulário Aberto por Texto) ---")
    print("  [38] yolov8s-world.pt | [39] yolov8m-world.pt | [40] yolov8l-world.pt | [41] yolov8x-world.pt")
    print("-" * 75)

    print("\n" + "=" * 75)
    print(" 🎯 ATENÇÃO: Insira sua escolha abaixo")
    print("=" * 75)

    # SELEÇÃO ADAPTADA (WIDGETS NO COLAB / INPUT TRADICIONAL SE LOCAL)
    if IN_COLAB:
        opcoes_dropdown = [(f"Opção {i}", str(i)) for i in range(1, 42)]
        dropdown = widgets.Dropdown(
            options=opcoes_dropdown,
            value='13',
            description='Modelo (Opção):',
            disabled=False,
        )
        botao = widgets.Button(description="Executar Detecção", button_style='success')
        
        def ao_clicar(b):
            clear_output(wait=True)
            executar_logica_execucao(dropdown.value)
            
        botao.on_click(ao_clicar)
        display(dropdown, botao)
    else:
        opcao = input("👉 Digite o número correspondente à sua escolha (1 a 41): ").strip()
        executar_logica_execucao(opcao)

if __name__ == "__main__":
    executar_simulacao()